# 10. XGBoost sin regional (`REG_*`) — Fase 6 equidad geográfica

`REG_1 REG BOGOTA` salía #3 en SHAP. Es legítimo (refleja dónde se concentra la atención), pero
por **equidad geográfica** preocupa que el modelo priorice leads de Bogotá por volumen de
población, no por riesgo individual. Este notebook prueba **eliminar** la regional (6 dummies
`REG_*`), de forma **acumulativa** con `tiene_avicena` ya excluida → **52 features**.

**Bar (modelo canónico, con regional):** XGBoost ganador → AUC-PR 0,0388, recall@top10% 0,699.

> **Reproducibilidad:** se **carga** el modelo sin-regional guardado por el pipeline y se evalúa
> en validación (exacto). Es un test de sensibilidad, **no** el modelo canónico.


In [1]:
import json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score

B = "bases"; RNG = 42
RES = json.load(open(f"{B}/_resultados_sin_avicena.json", encoding="utf-8"))

nat_va = pd.read_parquet(f"{B}/prediccion_mama_val_native.parquet")
REG_COLS = [c for c in nat_va.columns if c.startswith('REG_')]
DROP = ['tiene_avicena'] + REG_COLS
FEAT = [c for c in nat_va.columns if c not in ('key', 'label') and c not in DROP]
y_va = nat_va['label'].values.astype(int)
Xn_va = nat_va[FEAT].astype('float32').values
print(f"REG_* eliminadas: {REG_COLS}")
print(f"feats: {len(FEAT)} (eliminadas: tiene_avicena + {len(REG_COLS)} REG_*)")

def summarize(y, p, name):
    m = {'modelo': name, 'AUC_PR': float(average_precision_score(y, p)),
         'AUC_ROC': float(roc_auc_score(y, p))}
    for f in (0.005, 0.01, 0.05, 0.10):
        k = max(1, int(len(p) * f)); idx = np.argpartition(-p, k)[:k]
        m[f'recall@top{f*100:.1f}%'] = float(y[idx].sum() / y.sum())
    return m


REG_* eliminadas: ['REG_1 REG BOGOTA', 'REG_2 REG CALI', 'REG_3 REG BARRANQUILLA', 'REG_4 REG BUCARAMANGA', 'REG_5 REG MEDELLIN', 'REG_6 REG CENTRO ORIENTE']
feats: 52 (eliminadas: tiene_avicena + 6 REG_*)


## 1. Evaluación del modelo sin regional (52 feats)

In [2]:
model = joblib.load(f"{B}/modelo_fase6_XGBoost-sin-regional.joblib")
assert model.n_features_in_ == len(FEAT), (model.n_features_in_, len(FEAT))
p_va = model.predict_proba(Xn_va)[:, 1]
res = summarize(y_va, p_va, 'XGBoost-sin-avicena-sin-regional')
canon = RES['sin_regional']['validacion']
print(f"Confirmación modelo cargado: AUC-PR {res['AUC_PR']:.4f} (canónico {canon['AUC_PR']:.4f}) | "
      f"recall@top10% {res['recall@top10.0%']:.4f}")

win = RES['fase5b']['validacion'][0]   # XGBoost ganador (con regional)
def slim(d, label):
    return {'modelo': label, 'AUC_PR': d['AUC_PR'], 'AUC_ROC': d['AUC_ROC'],
            'recall@top1.0%': d['recall@top1.0%'], 'recall@top5.0%': d['recall@top5.0%'],
            'recall@top10.0%': d['recall@top10.0%']}
comp = pd.DataFrame([slim(res, 'SIN regional (52)'), slim(win, 'CON regional (58, ganador)')]).set_index('modelo')
pd.set_option('display.width', 200, 'display.max_columns', 20)
print("\n=== Validación 2023→2024 — sin regional vs ganador ===")
print(comp.to_string(float_format='{:.4f}'.format))
d_pr = res['AUC_PR'] - win['AUC_PR']; d_top10 = res['recall@top10.0%'] - win['recall@top10.0%']
print(f"\nDelta (sin − con regional) AUC-PR: {d_pr:+.4f} ({d_pr/win['AUC_PR']*100:+.1f}%) | "
      f"recall@top10%: {d_top10:+.4f}")
print("Veredicto:", "quitarla CUESTA (impacto NO despreciable)" if abs(d_pr) >= 0.002
      else "impacto despreciable")


Confirmación modelo cargado: AUC-PR 0.0310 (canónico 0.0310) | recall@top10% 0.6596

=== Validación 2023→2024 — sin regional vs ganador ===
                            AUC_PR  AUC_ROC  recall@top1.0%  recall@top5.0%  recall@top10.0%
modelo                                                                                      
SIN regional (52)           0.0310   0.8846          0.3162          0.5358           0.6596
CON regional (58, ganador)  0.0388   0.9007          0.3333          0.5680           0.6989

Delta (sin − con regional) AUC-PR: -0.0078 (-20.1%) | recall@top10%: -0.0393
Veredicto: quitarla CUESTA (impacto NO despreciable)


## 2. Distribución geográfica de los leads top-decil (equidad)

¿Quitar `REG_*` evita que Bogotá domine el top? Se mide la composición regional del top 10% de
riesgo del modelo SIN regional, usando las dummies `REG_*` solo como diagnóstico (no como input).


In [3]:
k = max(1, int(len(p_va) * 0.10))
top_idx = np.argpartition(-p_va, k)[:k]
share_top = nat_va[REG_COLS].iloc[top_idx].mean().sort_values(ascending=False)
share_all = nat_va[REG_COLS].mean().reindex(share_top.index)
tab = pd.DataFrame({'%_en_top10': share_top * 100, '%_en_poblacion': share_all * 100})
tab['sobre_representacion'] = tab['%_en_top10'] / tab['%_en_poblacion']
print("=== Composición regional del top-decil (modelo SIN regional) ===")
print(tab.to_string(float_format='{:.2f}'.format))
print("\nsobre_representación ~1 = proporcional a población; >1 = sobre-priorizada")
print("Aun quitando REG_*, otras features (consultas, labs, antropometría) son proxy de geografía.")


=== Composición regional del top-decil (modelo SIN regional) ===
                          %_en_top10  %_en_poblacion  sobre_representacion
REG_1 REG BOGOTA               44.22           36.61                  1.21
REG_2 REG CALI                 15.20           13.57                  1.12
REG_3 REG BARRANQUILLA         12.51           11.73                  1.07
REG_6 REG CENTRO ORIENTE       12.37           18.38                  0.67
REG_4 REG BUCARAMANGA           7.86           11.96                  0.66
REG_5 REG MEDELLIN              7.84            7.74                  1.01

sobre_representación ~1 = proporcional a población; >1 = sobre-priorizada
Aun quitando REG_*, otras features (consultas, labs, antropometría) son proxy de geografía.


## 3. Conclusión

Quitar la regional **cuesta** (AUC-PR cae ~20% y recall@top10% baja) y **no arregla** la
equidad: la geografía se cuela por features proxy (acceso → más datos → más señal). Por eso el
modelo canónico **conserva** la regional y la equidad se resuelve en la **asignación de leads**
(selección estratificada por regional, notebook 11), no borrando la feature.

Artefacto (solo sensibilidad): `bases/modelo_fase6_XGBoost-sin-regional.joblib`.
